# Lab 02 — CPU vs GPU Inference Benchmark
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chelsietao/ai-inference-lab/blob/main/notebooks/02_cpu_vs_gpu.ipynb)

> 用可重現的方法量測 CPU 與 GPU 的 latency 與 throughput,並理解**為什麼 GPU 不一定比較快**。

## 學習目標
1. 學會正確的 benchmark 流程:warm-up → 重複量測 → 統計(mean / median / P90 / P95)。
2. 理解 GPU 的 **asynchronous execution** 與 `torch.cuda.synchronize()`。
3. 區分 **latency**(單一請求多久)與 **throughput**(單位時間處理多少)。
4. 讀懂 speedup 與它的限制。

## 先備知識
完成 Lab 01 較佳。不需要 CUDA 知識。

## 需要 GPU 嗎?
CPU 部分一定會跑。**GPU 部分只在偵測到 CUDA GPU 時才會執行**;否則會明確顯示「GPU 比較不可用」,**不會產生任何假的 GPU 數據**。啟用方式:`Runtime` → `Change runtime type` → **T4 GPU**。

## 重要提醒
> ⚠️ 本專案的 GPU 程式碼已實作,但**尚未在本機開發環境以 CUDA 硬體驗證**;請在你自己的 Colab GPU runtime 執行並檢查結果。
> 不同 Colab session 可能分配到不同 GPU,**結果只能在同一份輸出內比較**,不要與別人的環境直接比較。

## 技術原理
- GPU kernel 是**非同步**啟動的:Python 呼叫立刻返回,GPU 在背景執行。沒有 synchronize 就計時,會低估 latency。
- **Warm-up**:前幾次執行包含 lazy initialization(CUDA context、cuDNN autotune、記憶體配置),不代表 steady-state。
- 小 workload 時,資料搬移與 kernel launch overhead 可能超過 GPU 運算省下的時間,所以 GPU 可能比 CPU 慢。

## 1. 參數

In [ ]:
BATCH_SIZE = 8          # 兩個 device 使用相同 workload 才能比較
WARMUP_ITERS = 10       # 丟棄的暖機次數
MEASURE_ITERS = 50      # 正式量測次數
GPU_PRECISION = "fp32"  # "fp32" 或 "fp16"(fp16 只在 CUDA 可用)
USE_PRETRAINED = True   # False = 隨機 weights(速度相同,但預測無意義;下載失敗時可用)

In [ ]:
# 安裝 AI Inference Lab 套件(只在尚未安裝時執行;PyTorch / torchvision 在 Colab 已內建)
import importlib.util, subprocess, sys
if importlib.util.find_spec("ai_inference_lab") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/chelsietao/ai-inference-lab.git@main"])

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import ai_inference_lab as lab
print("ai_inference_lab", lab.__version__)

## 2. Hardware 偵測
CPU 與 GPU **不是**等價的 benchmark 環境,所以先確認你有什麼。

In [ ]:
DEVICE = "auto"

hw = lab.detect_hardware(DEVICE)
print(hw.summary())

In [ ]:
loaded = lab.load_classifier(pretrained=USE_PRETRAINED)   # 載入時間不計入 latency

## 3. 執行 CPU vs GPU Benchmark
`compare_devices` 會先跑 CPU,若有 CUDA 再跑 GPU。每筆結果都包含 cold-start、warm-up 與量測次數的記錄。

In [ ]:
result = lab.compare_devices(loaded, BATCH_SIZE, WARMUP_ITERS, MEASURE_ITERS, GPU_PRECISION)
records = result["records"]
print(result["message"])
lab.summary_table(records)

## 4. Speedup 與解讀

In [ ]:
if result["speedup"] is not None:
    print(f"Speedup (CPU mean / GPU mean) = {result['speedup']:.2f}x  (batch={BATCH_SIZE})")
    print("> 1 表示 GPU 較快;< 1 表示 GPU 較慢。這只代表這個 model、這個 batch size、這個 session 的硬體。")
else:
    print("沒有 GPU 時不計算 speedup。")

## 5. 視覺化
圖表直接使用上面的量測結果(沒有任何寫死的數字)。

In [ ]:
import matplotlib.pyplot as plt
from ai_inference_lab.visualization import charts
charts.plot_device_latency(records, "results/cpu_vs_gpu_latency.png")
charts.plot_device_throughput(records, "results/cpu_vs_gpu_throughput.png")
charts.plot_memory(records, "results/cpu_vs_gpu_memory.png")
plt.show()

## 6. 匯出結果 (JSON / CSV)

In [ ]:
lab.save_json(records, "results/cpu_vs_gpu.json")
lab.save_csv(records, "results/cpu_vs_gpu.csv")
print("已儲存至 results/(Colab 暫存空間;要保留請在 Colab 左側檔案面板下載)")

## 預期觀察(不預先寫數字)
- **有 GPU 時**:在較大的 batch size 下 GPU 通常有較高 throughput;但在 `BATCH_SIZE = 1` 的小 workload,差距可能很小甚至 CPU 較快。**請自己改 `BATCH_SIZE` 重跑驗證。**
- Std(變異)大時代表結果不穩;請增加 `MEASURE_ITERS` 或重跑。
- **記憶體**:GPU 的 `gpu_peak_mb` 是 PyTorch tensors 的 peak allocated(weights + input + activations),**不包含** CUDA context 與 caching allocator 保留但未使用的空間,也不是系統總記憶體。CPU 在 v0.1 不量測記憶體。

## 為什麼 GPU 不一定比 CPU 快?
1. 工作量太小,無法填滿 GPU 的平行運算單元。
2. Kernel launch overhead 與 CPU→GPU 資料搬移佔比高。
3. 模型很小(MobileNetV3-Small 為輕量模型),CPU 也能高效執行。
4. Preprocessing 在 CPU 上,可能是整體瓶頸(本 benchmark 不含 preprocessing)。

## 常見錯誤
- 沒有 warm-up 就量測。
- 沒有 synchronize 就計時(GPU 結果會「好得不真實」)。
- 拿不同 batch size / precision / 硬體的結果當成同一種比較。
- 把一次量測當作結論。

## 真實產業應用
Inference 服務會針對延遲目標(latency SLA)與成本(每秒處理量)選擇硬體與 batch 策略;線上服務重視 latency,離線批次處理重視 throughput。

## 相關概念
Warm-up、Cold start、Percentile latency、Kernel launch overhead、Asynchronous execution。

## 官方參考資料
- [CUDA semantics — PyTorch](https://pytorch.org/docs/stable/notes/cuda.html)
- [torch.cuda.synchronize](https://pytorch.org/docs/stable/generated/torch.cuda.synchronize.html)
- [torch.cuda.max_memory_allocated](https://pytorch.org/docs/stable/generated/torch.cuda.max_memory_allocated.html)